# 🚀 OmniVoice TTS - Google Colab GPU T4 Worker

Notebook này cho phép bạn tận dụng **GPU NVIDIA Tesla T4 (16GB VRAM) miễn phí** trên Google Colab để tăng tốc sinh âm thanh cho dự án **self-tts**, đặc biệt phù hợp khi chạy máy tính cá nhân **không có card đồ hoạ rời (VGA)**.

### 🌟 Tính năng nâng cấp: Hỗ trợ Ngrok Static Domain
- **Link cố định vĩnh viễn:** Nhập **Ngrok Authtoken** và **Static Domain** (đăng ký miễn phí 1 tên miền tại [dashboard.ngrok.com](https://dashboard.ngrok.com/)).
- **Tự động kết nối 100%:** Chỉ cần điền vào file `backend/.env` trên máy bạn **1 LẦN DUY NHẤT**. Những lần sau chỉ cần bấm chạy Colab là ứng dụng tự động kết nối, không cần copy/paste lại link!
- **Dự phòng Cloudflare Tunnel:** Nếu chưa có tài khoản Ngrok, bạn vẫn có thể chọn chế độ `cloudflare` để sinh link tự động miễn phí.

---
### 📝 Hướng dẫn nhanh:
1. Vào menu **Runtime** -> **Change runtime type** -> Chọn **T4 GPU** -> Bấm **Save**.
2. (Khuyên dùng) Điền `NGROK_AUTHTOKEN` và `NGROK_STATIC_DOMAIN` vào ô cấu hình bên dưới.
3. Bấm nút **Chạy ô bên dưới (Run cell)** và đợi thông báo sẵn sàng!

In [ ]:
#@title ⚡ Khởi Động OmniVoice GPU Worker & Đường Hầm Tunnel (Bấm Run)
#@markdown Chọn phương thức kết nối về máy local của bạn:
#@markdown - **ngrok**: Khuyên dùng! Dùng **Static Domain cố định**, cấu hình file `.env` 1 lần dùng mãi mãi.
#@markdown - **cloudflare**: Miễn phí không cần tài khoản, link ngẫu nhiên mỗi lần chạy.
TUNNEL_METHOD = "ngrok" #@param ["ngrok", "cloudflare"]
NGROK_AUTHTOKEN = "" #@param {type:"string"}
NGROK_STATIC_DOMAIN = "" #@param {type:"string"}

import os
import sys
import time
import re
import subprocess
import urllib.request
import json

# 1. Kiểm tra GPU CUDA
print("🔍 [1/5] Đang kiểm tra card đồ hoạ GPU...")
gpu_check = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if gpu_check.returncode != 0:
    raise RuntimeError("❌ Chưa bật GPU! Vui lòng vào menu: Runtime -> Change runtime type -> Chọn T4 GPU rồi bấm chạy lại ô này.")
print("✅ Nhận diện GPU thành công!\n" + gpu_check.stdout.split("\n")[8])

# 2. Cài đặt các thư viện cần thiết
print("\n📦 [2/5] Đang cài đặt thư viện cần thiết (OmniVoice, Accelerate, Transformers, FastAPI, Uvicorn, PyNgrok)...")
reqs = [
    "omnivoice", "accelerate>=0.34.0", "webdataset", "safetensors", "transformers>=4.40.0",
    "soundfile", "librosa", "pydub", "scipy", "faster-whisper", "fastapi", "uvicorn[standard]",
    "python-multipart", "httpx", "pyngrok"
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + reqs, check=True)
print("✅ Đã cài đặt xong toàn bộ thư viện phụ thuộc!")

# 3. Tạo file colab_worker.py trực tiếp
print("\n⚙️ [3/5] Đang thiết lập mã nguồn OmniVoice Colab Worker...")
worker_code = '''import os
import io
import gc
import re
import logging
import tempfile
from pathlib import Path
import torch
import numpy as np
import soundfile as sf
from fastapi import FastAPI, HTTPException, UploadFile, File, Form
from fastapi.responses import Response, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from omnivoice import OmniVoice, VoiceClonePrompt

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] colab_worker — %(message)s", datefmt="%H:%M:%S")
logger = logging.getLogger("colab_worker")

app = FastAPI(title="OmniVoice Colab GPU Worker")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

_model = None
SAMPLE_RATE = 24000

def get_model():
    global _model
    if _model is None:
        dev_str = "cuda:0" if torch.cuda.is_available() else "cpu"
        logger.info(f"Đang tải OmniVoice lên thiết bị: {dev_str}...")
        _model = OmniVoice.from_pretrained("k2-fsa/OmniVoice", device=dev_str)
        logger.info("✅ Tải mô hình OmniVoice thành công!")
    return _model

@app.on_event("startup")
def startup():
    get_model()

@app.get("/api/remote/health")
def health():
    gpu_ok = torch.cuda.is_available()
    gpu_name = torch.cuda.get_device_name(0) if gpu_ok else "CPU"
    vram = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 1) if gpu_ok else 0
    return {"status": "ok", "gpu_available": gpu_ok, "gpu_name": gpu_name, "vram_total_gb": vram, "provider": "Google Colab (T4 16GB)"}

@app.post("/api/remote/prompt")
async def create_prompt(audio_file: UploadFile = File(...), ref_text: str = Form(None)):
    model = get_model()
    suffix = Path(audio_file.filename or "ref.wav").suffix or ".wav"
    with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp:
        tmp.write(await audio_file.read())
        tmp_path = tmp.name
    try:
        t_ref = ref_text.strip() if ref_text and ref_text.strip() else None
        prompt = model.create_voice_clone_prompt(ref_audio=tmp_path, ref_text=t_ref)
        with tempfile.NamedTemporaryFile(delete=False, suffix=".pt") as tmp_pt:
            prompt.save(tmp_pt.name)
            tmp_pt_path = tmp_pt.name
        with open(tmp_pt_path, "rb") as pf:
            content = pf.read()
        try: os.remove(tmp_pt_path)
        except OSError: pass
        return Response(content=content, media_type="application/octet-stream")
    finally:
        try: os.remove(tmp_path)
        except OSError: pass
        if torch.cuda.is_available(): torch.cuda.empty_cache()

def clean_vietnamese_text(text: str) -> str:
    import re
    text = re.sub(r"<[^>]+>", "", text)
    text = re.sub(r"[""'']", "", text)
    text = re.sub(r"[\s\t]+", " ", text)
    return text.strip()

def split_into_chunks(text: str, max_chars: int = 450) -> list[str]:
    import re
    text = clean_vietnamese_text(text)
    if not text: return []
    if len(text) <= max_chars: return [text]
    parts = re.split(r"([.!?…]+(?:\s+|$)|;\s+|:\s+)", text)
    sentences = []
    i = 0
    while i < len(parts):
        s = parts[i]
        if i + 1 < len(parts) and re.match(r"^[.!?…;: ]+$", parts[i+1]):
            s += parts[i+1]
            i += 2
        else:
            i += 1
        s = s.strip()
        if s: sentences.append(s)
    chunks = []
    cur = ""
    for s in sentences:
        if not cur:
            cur = s
        elif len(cur) + 1 + len(s) <= max_chars:
            cur += " " + s
        else:
            chunks.append(cur)
            cur = s
    if cur: chunks.append(cur)
    return chunks

@app.post("/api/remote/generate")
async def generate_audio_endpoint(
    text: str = Form(...),
    mode: str = Form("clone"),
    num_step: int = Form(16),
    cfg_value: float = Form(2.0),
    speed: float = Form(1.0),
    seed: int = Form(None),
    instruct: str = Form(None),
    ref_text: str = Form(None),
    prompt_file: UploadFile = File(None),
    ref_audio_file: UploadFile = File(None),
):
    _model = get_model()
    voice_clone_prompt = None
    tmp_ref_path = None
    try:
        if prompt_file is not None:
            pt_bytes = await prompt_file.read()
            with tempfile.NamedTemporaryFile(delete=False, suffix=".pt") as tmp_pt:
                tmp_pt.write(pt_bytes)
                tmp_pt_path = tmp_pt.name
            try:
                dev_str = "cuda:0" if torch.cuda.is_available() else "cpu"
                voice_clone_prompt = VoiceClonePrompt.load(tmp_pt_path, map_location=dev_str)
            except Exception as e:
                logger.warning(f"Không thể đọc prompt_file: {e}")
            finally:
                try: os.remove(tmp_pt_path)
                except OSError: pass

        if ref_audio_file is not None:
            sfx = Path(ref_audio_file.filename or "ref.wav").suffix or ".wav"
            with tempfile.NamedTemporaryFile(delete=False, suffix=sfx) as tmp:
                tmp.write(await ref_audio_file.read())
                tmp_ref_path = tmp.name

        chunks = split_into_chunks(text, max_chars=450)
        logger.info(f"⚡ [Colab GPU] Xử lý {len(chunks)} chunks | num_step={num_step} | Text: '{text[:40]}…'")
        all_audios = []
        design_voice_clone_prompt = None
        with torch.inference_mode():
            for chunk in chunks:
                gen_kwargs = {"text": chunk, "language": "vi", "num_step": num_step, "guidance_scale": cfg_value, "normalize_text": False, "speed": speed}
                if mode == "clone":
                    if voice_clone_prompt is not None: gen_kwargs["voice_clone_prompt"] = voice_clone_prompt
                    elif tmp_ref_path:
                        gen_kwargs["ref_audio"] = tmp_ref_path
                        if ref_text and ref_text.strip(): gen_kwargs["ref_text"] = clean_vietnamese_text(ref_text)
                elif mode == "design":
                    if design_voice_clone_prompt is not None: gen_kwargs["voice_clone_prompt"] = design_voice_clone_prompt
                    elif instruct: gen_kwargs["instruct"] = instruct
                audio_list = _model.generate(**gen_kwargs)
                if audio_list and len(audio_list) > 0:
                    anp = np.array(audio_list[0], dtype=np.float32)
                    if anp.ndim > 1: anp = anp.squeeze()
                    all_audios.append(anp)
        if not all_audios: raise HTTPException(status_code=500, detail="Không sinh được âm thanh")
        silence_array = np.zeros(int(SAMPLE_RATE * 0.22), dtype=np.float32)
        final_pieces = []
        fade_len = int(SAMPLE_RATE * 0.01)
        for i, a in enumerate(all_audios):
            if len(a) > fade_len * 2:
                a[:fade_len] *= np.linspace(0, 1, fade_len, dtype=np.float32)
                a[-fade_len:] *= np.linspace(1, 0, fade_len, dtype=np.float32)
            final_pieces.append(a)
            if i < len(all_audios) - 1: final_pieces.append(silence_array)
        final_audio = np.concatenate(final_pieces)
        wav_buf = io.BytesIO()
        sf.write(wav_buf, final_audio, SAMPLE_RATE, format="WAV")
        wav_buf.seek(0)
        return Response(content=wav_buf.getvalue(), media_type="audio/wav")
    finally:
        if tmp_ref_path: 
            try: os.remove(tmp_ref_path)
            except OSError: pass
        if torch.cuda.is_available(): torch.cuda.empty_cache()

if __name__ == '__main__':
    import uvicorn
    uvicorn.run(app, host='127.0.0.1', port=8000)
'''
with open("colab_worker.py", "w", encoding="utf-8") as f:
    f.write(worker_code)

# 4. Khởi động server ngầm và ghi log
print("\n🚀 [4/5] Đang khởi động OmniVoice GPU Worker & nạp mô hình vào VRAM...")
log_file = open("worker.log", "w", encoding="utf-8")
server_proc = subprocess.Popen([sys.executable, "-u", "colab_worker.py"], stdout=log_file, stderr=subprocess.STDOUT)

# Chờ server sẵn sàng bằng cách poll health endpoint (tối đa 90s)
is_ready = False
print("⏳ Đang tải trọng số mô hình k2-fsa/OmniVoice (vui lòng đợi ~30-60 giây)...", end="", flush=True)
for _ in range(45):
    if server_proc.poll() is not None:
        break
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/api/remote/health", timeout=2) as resp:
            if resp.status == 200:
                data = json.loads(resp.read().decode())
                is_ready = True
                print(f"\n✅ Server đã sẵn sàng! [{data.get('gpu_name')} - {data.get('vram_total_gb')}GB VRAM]")
                break
    except Exception:
        pass
    print(".", end="", flush=True)
    time.sleep(2)

if not is_ready:
    log_file.close()
    with open("worker.log", "r", encoding="utf-8") as f:
        err_output = f.read()
    raise RuntimeError(f"❌ Server không khởi động được! Chi tiết log lỗi:\n\n{err_output}")

# 5. Khởi chạy Đường hầm Tunnel (Ngrok Static Domain hoặc Cloudflare Tunnel)
tunnel_url = None

if TUNNEL_METHOD == "ngrok" and NGROK_AUTHTOKEN.strip():
    print("\n🌐 [5/5] Đang kết nối đường hầm bảo mật qua Ngrok...")
    try:
        from pyngrok import ngrok, conf
        ngrok.set_auth_token(NGROK_AUTHTOKEN.strip())
        domain_val = NGROK_STATIC_DOMAIN.strip() if NGROK_STATIC_DOMAIN.strip() else None
        if domain_val:
            print(f"🔗 Đang kết nối tới tên miền tĩnh Ngrok: {domain_val} ...")
            tunnel = ngrok.connect(8000, "http", domain=domain_val)
        else:
            print("🔗 Đang tạo đường hầm Ngrok ngẫu nhiên...")
            tunnel = ngrok.connect(8000, "http")
        tunnel_url = tunnel.public_url
    except Exception as ne:
        print(f"⚠️ Lỗi kết nối Ngrok ({ne}). Đang tự động chuyển sang Cloudflare Tunnel dự phòng...")

if not tunnel_url:
    print("\n🌐 [5/5] Đang kết nối đường hầm Cloudflare Tunnel dự phòng...")
    subprocess.run(["wget", "-q", "-nc", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"], check=True)
    subprocess.run(["dpkg", "-i", "-E", "cloudflared-linux-amd64.deb"], capture_output=True)
    tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    start_time = time.time()
    while time.time() - start_time < 30:
        line = tunnel_proc.stdout.readline()
        if not line: continue
        match = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)
        if match:
            tunnel_url = match.group(0)
            break

if tunnel_url:
    is_static = bool(TUNNEL_METHOD == "ngrok" and NGROK_STATIC_DOMAIN.strip() and NGROK_STATIC_DOMAIN.strip() in tunnel_url)
    print("\n" + "="*70)
    print("🎉 GOOGLE COLAB GPU (T4 16GB) ĐÃ SẴN SÀNG HOẠT ĐỘNG!")
    print(f"👉 URL kết nối: {tunnel_url}")
    print("="*70)
    if is_static:
        print("\n✨ BẠN ĐANG DÙNG NGROK STATIC DOMAIN CỐ ĐỊNH!")
        print("💡 Chỉ cần cấu hình file backend/.env trên máy bạn 1 LẦN DUY NHẤT:")
        print("-"*70)
        print("USE_COLAB_GPU=true")
        print(f"COLAB_API_URL={tunnel_url}")
        print("-"*70)
        print("🚀 Những ngày sau chỉ cần mở Colab bấm Chạy, web sẽ TỰ ĐỘNG KẾT NỐI mà không bao giờ cần sửa lại .env!\n")
    else:
        print("\n📋 HÃY SAO CHÉP 2 DÒNG DƯỚI ĐÂY DÁN VÀO FILE backend/.env TRÊN MÁY BẠN:")
        print("-"*70)
        print("USE_COLAB_GPU=true")
        print(f"COLAB_API_URL={tunnel_url}")
        print("-"*70 + "\n")
    print("💡 Giữ tab Google Colab này mở trong suốt quá trình sử dụng hệ thống.\n")
    try:
        if TUNNEL_METHOD == "ngrok" and NGROK_AUTHTOKEN.strip() and "ngrok" in tunnel_url:
            from pyngrok import ngrok
            ngrok.get_ngrok_process().proc.wait()
        else:
            tunnel_proc.wait()
    except KeyboardInterrupt:
        print("🛑 Đã dừng worker.")
else:
    print("⚠️ Không tìm thấy URL kết nối. Vui lòng thử khởi động lại ô này.")
